# 05 — Modèles de base, comparaison des couches et ablation

**Objectif** : mesurer ce que chaque couche de caractéristiques (URL, contenu, externe) apporte à la détection du phishing, avec les 4 classifieurs du cours (KNN, arbre de décision, Random Forest, SVM).

**Règles suivies dans tout le notebook :**
- On travaille **uniquement sur `train.csv`** (9 003 URL). Le test verrouillé (`test_LOCKED.csv`) n'est pas ouvert.
- Validation croisée **groupée par site** : `GroupKFold(5)` sur la colonne `group` (définie dans `04_split`), pour qu'un même site ne soit jamais à la fois en entraînement et en validation.
- Classe positive : `y = 1` = **phishing**. Critère principal : **F1** (pas l'accuracy).
- `random_state=42` partout.

In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("../data/processed/train.csv")

## 1. Les 3 couches de caractéristiques

Les 87 colonnes de Hannousse sont réparties en 3 couches, délimitées par le nom de leur première et dernière colonne :

| Couche | Colonnes | Nombre | Ce qu'elle mesure | Coût en production |
| --- | --- | --- | --- | --- |
| **URL** | `length_url` → `statistical_report` | 56 | forme de l'URL (longueur, caractères spéciaux, sous-domaines, mots suspects…) | gratuit, instantané |
| **Contenu** | `nb_hyperlinks` → `domain_with_copyright` | 24 | structure de la page HTML (liens, formulaires, iframes…) | télécharger la page |
| **Externe** | `whois_registered_domain` → `page_rank` | 7 | WHOIS, index Google, trafic, PageRank | services tiers (lents, parfois disparus) |

`url`, `status` et les colonnes de groupe ne sont **pas** des caractéristiques : ce sont des identifiants.

In [3]:
cols = list(train.columns)
URL_COLS = cols[cols.index("length_url") : cols.index("statistical_report") + 1]
CONTENT_COLS = cols[cols.index("nb_hyperlinks") : cols.index("domain_with_copyright") + 1]
EXTERNAL_COLS = cols[cols.index("whois_registered_domain") : cols.index("page_rank") + 1]

print(len(URL_COLS), len(CONTENT_COLS), len(EXTERNAL_COLS))

56 24 7


In [4]:
X = train[URL_COLS + CONTENT_COLS + EXTERNAL_COLS]
y = (train["status"] == "phishing").astype(int)   # 1 = phishing
groups = train["group"]

print(X.shape, y.mean().round(3), groups.nunique())

(9003, 87) 0.492 5948


## 2. Premier modèle : Random Forest sur la couche URL

Random Forest avec ses réglages par défaut, évalué par validation croisée groupée (5 plis). On obtient un score par pli : la **moyenne** donne la performance attendue, l'**écart-type** sa stabilité.

In [5]:
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier


In [6]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

In [7]:
scores = cross_validate(rf, X[URL_COLS], y, groups=groups, cv=cv,scoring=["f1", "precision", "recall", "roc_auc"])

In [8]:
pd.DataFrame(scores).drop(columns=["fit_time", "score_time"]).agg(["mean", "std"]).round(4)

,test_f1,test_precision,test_recall,test_roc_auc
mean,0.8784,0.8729,0.8843,0.9521
std,0.0246,0.0283,0.0250,0.0125


**Résultat** : F1 = **0,878** (± 0,025), ROC-AUC = 0,952. La seule forme de l'URL, sans page ni popularité, apporte déjà un vrai signal. L'écart-type d'environ 2,5 points signifie qu'une différence de moins de ~2 points entre deux modèles n'est pas forcément réelle.

## 3. Pourquoi découper par site : `GroupKFold` contre `KFold` aléatoire

Même modèle, mêmes données, seul le découpage change. Avec `KFold` (ce que font la plupart des publications), les URL d'un même site sont dispersées entre entraînement et validation : le modèle peut « reconnaître » un site déjà vu.

In [9]:
from sklearn.model_selection import KFold

In [10]:
cv_random = KFold(n_splits=5, shuffle=True, random_state=42)

In [11]:
scores_random = cross_validate(rf, X[URL_COLS], y, cv=cv_random,scoring=["f1", "precision", "recall", "roc_auc"])

In [12]:
comparison = pd.DataFrame({
    "par site (GroupKFold)": pd.DataFrame(scores).mean(),
    "aléatoire (KFold)": pd.DataFrame(scores_random).mean(),
}).drop(["fit_time", "score_time"]).round(4)
comparison["écart"] = comparison["aléatoire (KFold)"] - comparison["par site (GroupKFold)"]
comparison

,par site (GroupKFold),aléatoire (KFold),écart
test_f1,0.8784,0.9082,0.0298
test_precision,0.8729,0.9112,0.0383
test_recall,0.8843,0.9053,0.0210
test_roc_auc,0.9521,0.9675,0.0154


**Résultat** : le découpage aléatoire gonfle le F1 de **+3,0 points** (0,878 → 0,908) et la précision de +3,8 points, avec exactement le même modèle. Ce gain vient de la mémorisation des sites, pas de la détection. C'est plus que ce que le stacking apporte en général (1–2 points) : **le choix du découpage pèse plus que le choix du modèle.** Tous les résultats suivants utilisent `GroupKFold`.

## 4. Les 4 classifieurs sur la couche URL

Réglages par défaut. `StandardScaler` pour KNN et SVM (ils calculent des distances), pas pour l'arbre et le RF (ils comparent chaque colonne à un seuil). La normalisation est dans un `Pipeline` : elle est apprise sur les plis d'entraînement seulement.

In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [14]:
models = {
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Arbre": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "SVM": make_pipeline(StandardScaler(), SVC(random_state=42)),
}

In [15]:
rows = {}
for name, model in models.items():
    s = cross_validate(model, X[URL_COLS], y, groups=groups, cv=cv,
                       scoring=["f1", "precision", "recall", "roc_auc"])
    s = pd.DataFrame(s)
    rows[name] = {"F1": s["test_f1"].mean(), "F1 std": s["test_f1"].std(),
                  "précision": s["test_precision"].mean(), "rappel": s["test_recall"].mean(),
                  "ROC-AUC": s["test_roc_auc"].mean()}

In [16]:
pd.DataFrame(rows).T.round(4).sort_values("F1", ascending=False)

,F1,F1 std,précision,rappel,ROC-AUC
RF,0.8784,0.0246,0.8729,0.8843,0.9521
SVM,0.8683,0.0314,0.8751,0.8619,0.9410
KNN,0.8367,0.0524,0.8723,0.8050,0.9038
Arbre,0.8208,0.0465,0.8300,0.8129,0.8268


**Résultat** (F1) : RF **0,878** > SVM 0,868 > KNN 0,837 > Arbre 0,821.
- RF en tête, comme dans l'état de l'art, mais SVM est à 1 point, en dessous de l'écart-type : pas de différence significative.
- L'arbre seul sur-apprend (pas de limite de profondeur, ROC-AUC 0,827) ; le RF (300 arbres moyennés) gagne 6 points.

## 5. Les 4 classifieurs × les 3 couches (12 modèles)

Avant d'utiliser la couche externe, on traite les codes « inconnu » (décision prise dans `03_mendeley_eda`) :
- `domain_age` et `domain_registration_length` < 0 (-1, -2, -12 = échec WHOIS) → `NaN` ;
- dans le `Pipeline` : `SimpleImputer(strategy="median", add_indicator=True)` remplace par la médiane **des plis d'entraînement** et ajoute une colonne « était inconnu ».

In [17]:
from sklearn.impute import SimpleImputer

In [18]:
# -1, -2, -12 = « inconnu » (échec WHOIS), pas un vrai nombre -> valeur manquante
X = X.copy()
for col in ["domain_age", "domain_registration_length"]:
    X[col] = X[col].mask(X[col] < 0)

def make_models():
    impute = lambda: SimpleImputer(strategy="median", add_indicator=True)
    return {
        "KNN": make_pipeline(impute(), StandardScaler(), KNeighborsClassifier()),
        "Arbre": make_pipeline(impute(), DecisionTreeClassifier(random_state=42)),
        "RF": make_pipeline(impute(), RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
        "SVM": make_pipeline(impute(), StandardScaler(), SVC(random_state=42)),
    }

LAYERS = {"URL": URL_COLS, "Contenu": CONTENT_COLS, "Externe": EXTERNAL_COLS}

results = []
for layer, layer_cols in LAYERS.items():
    for name, model in make_models().items():
        s = pd.DataFrame(cross_validate(model, X[layer_cols], y, groups=groups, cv=cv,
                                        scoring=["f1", "roc_auc"]))
        results.append({"couche": layer, "modèle": name, "F1": s["test_f1"].mean(),
                        "F1 std": s["test_f1"].std(), "ROC-AUC": s["test_roc_auc"].mean()})

results = pd.DataFrame(results)
results.pivot(index="modèle", columns="couche", values="F1").round(4)

couche,Contenu,Externe,URL
modèle,,,
Arbre,0.7616,0.8616,0.8208
KNN,0.7973,0.8800,0.8368
RF,0.8006,0.8992,0.8784
SVM,0.8203,0.8738,0.8683


**Résultat** (F1) :

| Modèle | URL | Contenu | Externe |
| --- | --- | --- | --- |
| KNN | 0,837 | 0,797 | 0,880 |
| Arbre | 0,821 | 0,762 | 0,862 |
| RF | **0,878** | 0,801 | **0,899** |
| SVM | 0,868 | **0,820** | 0,874 |

- **Les 7 caractéristiques externes battent les 56 de l'URL.** Elles mesurent la **popularité** du site (la règle `google_index` seule donnait déjà 0,869), pas le phishing.
- Le contenu seul est la couche la plus faible.
- Le meilleur algorithme change selon la couche (RF, SVM) : argument pour le **stacking** (meilleur modèle par couche, puis combinaison).

## 6. Ablation : URL → + Contenu → + Externe

On ajoute les couches une à une, dans l'ordre de leur coût en production, pour mesurer ce que **chacune ajoute** aux précédentes.

In [19]:
ABLATION = {
    "1. URL": URL_COLS,
    "2. URL + Contenu": URL_COLS + CONTENT_COLS,
    "3. URL + Contenu + Externe": URL_COLS + CONTENT_COLS + EXTERNAL_COLS,
}

abl = []
for step, step_cols in ABLATION.items():
    for name, model in make_models().items():
        s = pd.DataFrame(cross_validate(model, X[step_cols], y, groups=groups, cv=cv,
                                        scoring=["f1", "roc_auc"]))
        abl.append({"étape": step, "modèle": name, "F1": s["test_f1"].mean(),
                    "F1 std": s["test_f1"].std(), "ROC-AUC": s["test_roc_auc"].mean()})

abl = pd.DataFrame(abl)
abl.pivot(index="modèle", columns="étape", values="F1").round(4)


étape,1. URL,2. URL + Contenu,3. URL + Contenu + Externe
modèle,,,
Arbre,0.8208,0.8276,0.9101
KNN,0.8368,0.8528,0.8966
RF,0.8784,0.9168,0.9546
SVM,0.8683,0.9054,0.9392


**Résultat** (F1) :

| Modèle | URL | + Contenu | + Externe |
| --- | --- | --- | --- |
| KNN | 0,837 | 0,853 | 0,897 |
| Arbre | 0,821 | 0,828 | 0,910 |
| RF | 0,878 | **0,917** | **0,955** |
| SVM | 0,868 | 0,905 | 0,939 |

- **Le contenu apporte +3,8 points au RF** : seul il est faible, mais il complète l'URL (l'URL dit *où* est la page, le contenu *ce qu'elle fait*).
- **L'externe ajoute encore +3,8 points, mais grâce à la popularité** : l'arbre seul gagne +8,3 points en s'appuyant sur `google_index`.
- **Chiffre de référence retenu : RF, URL + Contenu, F1 = 0,917.** Le 0,955 n'est pas transposable (popularité, services disparus, biais de sélection des légitimes).